# Example: a liquid bridge between two beads

A droplet wets two nearly touching spherical beads and forms a **capillary
bridge**: the shape of minimal surface energy at fixed liquid volume, meeting
each bead at a prescribed **contact angle** $\theta$. Bridges like this hold wet
granular media together; the force they exert depends on the gap between the
beads and on $\theta$.

The beads have radius 1 and the liquid–gas surface tension is 1, so lengths are
in bead radii, energies in surface tension × radius², and forces in surface
tension × radius. The adjustable parameters:

* `theta`: the contact angle in degrees, measured inside the liquid
  (below 90°: wetting);
* `gap`: the distance between the bead surfaces (0: touching);
* `volume`: the liquid volume.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pyvista as pv
import pysurfaceevolver as pyse
from pysurfaceevolver import constraints as C

pv.set_jupyter_backend("static")    # "trame" in Jupyter for interactive views
pv.global_theme.window_size = [600, 450]

## The model

Only the liquid–gas surface is made of facets. The beads are **constraints**:
the two contact lines live on the spheres
$x^2 + y^2 + (z \pm c)^2 = 1$, with $c = 1 + \mathrm{gap}/2$, and slide freely on
them. The wetted part of each bead is not meshed; it enters as integrals along
the contact line, which Evolver adds to the edges on a constraint (as in the
`mound.fe` sample, there on a plane):

* **Energy.** Wetting a unit area of bead replaces solid–gas by solid–liquid
  surface, a change of $\gamma_{SL} - \gamma_{SG} = -\cos\theta$ (Young's law).
  So the energy is the liquid area plus $-\cos\theta$ times the wetted area of
  each bead.
* **Volume.** Evolver computes a body's volume from its facets as
  $\int z\, n_z\, dA$; the wetted caps close the liquid's boundary, so their part of
  that integral is added too.

[`pyse.constraints.sphere`](api.md) writes both integrals. Over the azimuth $\phi$
about the $z$ axis, with $h$ the height above a bead's centre, the wetted area is
$\oint R\,h\,d\phi$ up to the pole facing the gap (the integrand is singular at the
poles), and the bead's share of $\int z\, n_z\, dA$ is $\oint (c_z h^2/2 + h^3/3)\, d\phi$.
Told which pole is wet (`wet_poles`), the builder returns what those integrals
leave out: a constant for the body's `volconst`, and the wetted area
`area_constant` ($2\pi R^2$ per bead) behind a constant energy
$-\cos\theta \cdot 2\pi R^2$, which Evolver's reported energy leaves out (it
doesn't change with the gap, so forces don't see it). The contact angle is the
datafile parameter `theta`, so it can be changed on a loaded surface.

In [ ]:
def beads(gap, radius=1.0):
    # the two beads, wetted around the poles that face each other; the contact
    # angle is the datafile parameter theta
    c = radius + gap/2
    return {1: C.sphere((0, 0, -c), radius, contact_angle="theta", wet_poles=("north",)),
            2: C.sphere((0, 0, c), radius, contact_angle="theta", wet_poles=("south",))}

def start_radius(gap, volume, radius=1.0):
    # radius a of a cylinder between the beads holding `volume`:
    # V(a) = 2 pi [c a^2 - 2/3 (R^3 - (R^2 - a^2)^(3/2))]
    c = radius + gap/2
    lo, hi = 0.0, 0.999*radius
    for _ in range(60):
        a = (lo + hi)/2
        v = 2*np.pi*(c*a**2 - 2/3*(radius**3 - (radius**2 - a**2)**1.5))
        lo, hi = (a, hi) if v < volume else (lo, a)
    return a

def bridge_datafile(gap, theta, volume, radius=1.0, n=8, rings=3):
    # a cylinder of liquid between the beads, its ends on the bead surfaces
    c = radius + gap/2
    a = start_radius(gap, volume, radius)
    z0 = -c + np.sqrt(radius**2 - a**2)
    phi = 2*np.pi*np.arange(n)/n
    vertices = np.vstack([np.column_stack([a*np.cos(phi), a*np.sin(phi),
                                           np.full(n, z0 - 2*z0*i/rings)])
                          for i in range(rings + 1)])
    faces = [[i*n + j, i*n + (j + 1) % n, (i + 1)*n + (j + 1) % n, (i + 1)*n + j]
             for i in range(rings) for j in range(n)]     # normals point out of the liquid
    cons = beads(gap, radius)
    return pyse.make_datafile(
        vertices, faces,
        bodies=[pyse.Body(faces=range(len(faces)), volume=volume,
                          volconst=cons[1].volconst + cons[2].volconst)],
        parameters={"radius": radius, "gap": gap, "theta": theta},
        constraints=cons,
        vertex_constraints={1: range(n), 2: range(rings*n, (rings + 1)*n)})

## Relaxing

The relaxation refines the mesh three times (to about 6000 facets), with
gradient steps in between, and ends with Newton steps. Every ten steps it
equiangulates and averages vertices (`ev.relax(..., tidy=10)`): without that,
facets at the contact line get thin and collapse when the contact line moves.

Very low contact angles at larger gaps make the contact line travel far over
the beads; there the bridge is relaxed at 40° first and `theta` is then lowered
in steps of 10°, relaxing after each.

In [ ]:
def settle(ev, callback=None):
    # relax a refined surface: gradient steps with mesh upkeep (equiangulation
    # and vertex averaging every 10, then every 5 steps), then Newton
    ev.relax(max_iter=80, tidy=10, callback=callback, every=10)
    ev.relax(max_iter=15, tidy=5)
    ev.iterate(5)
    ev.newton(3)
    if callback: callback()

def relax_bridge(ev, levels=3, callback=None):
    ev.iterate(20)
    for _ in range(levels):
        ev.refine()
        ev.relax(max_iter=80, tidy=10, callback=callback, every=10)
    settle(ev, callback)

def liquid_bridge(gap=0.2, theta=40.0, volume=0.3, callback=None):
    ev = pyse.Evolver()
    ev.load_string(bridge_datafile(gap, max(theta, 40.0), volume))
    relax_bridge(ev, callback=callback)
    while ev.parameters["theta"] > theta + 1e-9:
        ev.parameters["theta"] = max(theta, ev.parameters["theta"] - 10)
        settle(ev, callback)
    return ev

Here is a bridge relaxing from its starting cylinder. In Jupyter (trame backend)
the live view redraws during the run; these docs show its first frame, and the
relaxed bridge below.

In [ ]:
def add_beads(plotter, gap, opacity=1.0):
    c = 1 + gap/2
    for z in (-c, c):
        plotter.add_mesh(pv.Sphere(radius=1, center=(0, 0, z), theta_resolution=60,
                                   phi_resolution=60), color="sandybrown", opacity=opacity)
    plotter.camera_position = [(3.2, -3.2, 1.2), (0, 0, 0), (0, 0, 1)]

gap, theta, volume = 0.2, 40.0, 0.3

ev = pyse.Evolver()
ev.load_string(bridge_datafile(gap, theta, volume))
view = ev.live_view(color="lightskyblue", opacity=0.9)
add_beads(view.plotter, gap)
relax_bridge(ev, callback=view.update)

In [ ]:
def show_bridge(ev):
    plotter = pv.Plotter()
    add_beads(plotter, ev.parameters["gap"])
    plotter.add_mesh(ev.mesh().to_pyvista(), color="lightskyblue", smooth_shading=True)
    plotter.show()

show_bridge(ev)
ev

## Checking the result

The contact angle is not imposed on the mesh: it comes out of the energy
balance at the contact line. Measuring it is a check of the whole model: at each
contact-line vertex, the angle between the liquid surface's normal (out of the
liquid) and the bead's outward normal.

The pressure is the Lagrange multiplier of the volume constraint. The force
between the beads follows from the neck, the narrowest circle in the mid-plane:
there the surface is vertical by symmetry, so the force transmitted through
that plane is $F = 2\pi r_n \gamma - \pi r_n^2\, \Delta p$ (positive: the beads
attract). It must also equal the change of energy with the gap at fixed volume,
$F = dE/d\,\mathrm{gap}$.

In [ ]:
def contact_angles(ev):
    m = ev.mesh()
    c = 1 + ev.parameters["gap"]/2
    a, b, cc = (m.vertices[m.facets[:, i]] for i in range(3))
    out_of_liquid = np.where(m.facet_bodies[:, 0] == 1, 1.0, -1.0)
    normals = np.cross(b - a, cc - a) * out_of_liquid[:, None]   # area-weighted
    vertex_normals = np.zeros_like(m.vertices)
    for k in range(3):
        np.add.at(vertex_normals, m.facets[:, k], normals)
    angles = []
    for k, z in ((1, -c), (2, c)):
        on = ev.on_constraint(k)
        bead = (m.vertices[on] - [0, 0, z])                 # bead normals (radius 1)
        nl = vertex_normals[on] / np.linalg.norm(vertex_normals[on], axis=1)[:, None]
        angles.append(np.degrees(np.arccos(np.clip((nl*bead).sum(1), -1, 1))))
    return np.concatenate(angles)

def neck_radius(ev):
    # mean radius where the liquid surface crosses the mid-plane z = 0
    m = ev.mesh()
    p, q = m.vertices[m.edges[:, 0]], m.vertices[m.edges[:, 1]]
    crossing = (p[:, 2] < 0) != (q[:, 2] < 0)
    t = p[crossing, 2] / (p[crossing, 2] - q[crossing, 2])
    x = p[crossing] + t[:, None]*(q[crossing] - p[crossing])
    return np.hypot(x[:, 0], x[:, 1]).mean()

def force(ev):
    rn, p = neck_radius(ev), ev.bodies().pressure[0]
    return 2*np.pi*rn - np.pi*rn**2*p

angles = contact_angles(ev)
print(f"contact angle {angles.mean():.2f}° ± {angles.std():.2f}° (theta = {theta}°)")
print(f"pressure {ev.bodies().pressure[0]:.4f}, neck radius {neck_radius(ev):.4f}, "
      f"force {force(ev):.4f}, unstable directions {ev.eigen_counts().negative}")

h = 0.01
dE = (liquid_bridge(gap + h, theta, volume).total_energy
      - liquid_bridge(gap - h, theta, volume).total_energy) / (2*h)
print(f"dE/dgap {dE:.4f}")

The measured angle is within about a degree of `theta` (it is estimated from
flat facets, and is less accurate at very low and very high angles), and the two
forces agree to a fraction of a percent. `eigenprobe 0` counts the negative
eigenvalues of the Hessian: none, so this bridge is stable.

## Profiles for several contact angles

The more the liquid wets the beads, the further it spreads over them and the
narrower its neck. Cross-sections through the plane $y = 0$:

In [ ]:
def profile(ev):
    # (x, z) where the liquid surface crosses the half-plane y = 0, x > 0
    m = ev.mesh()
    p, q = m.vertices[m.edges[:, 0]], m.vertices[m.edges[:, 1]]
    crossing = ((p[:, 1] < 0) != (q[:, 1] < 0)) & (p[:, 0] + q[:, 0] > 0)
    t = p[crossing, 1] / (p[crossing, 1] - q[crossing, 1])
    x = p[crossing] + t[:, None]*(q[crossing] - p[crossing])
    order = np.argsort(x[:, 2])
    return x[order, 0], x[order, 2]

fig, ax = plt.subplots(figsize=(6, 4))
c = 1 + gap/2
s = np.linspace(0, 2*np.pi, 200)
for z in (-c, c):
    ax.fill(np.cos(s), z + np.sin(s), color="sandybrown", alpha=0.5, lw=0)
for th, color in zip((20, 40, 60, 80), plt.cm.viridis(np.linspace(0, 0.9, 4))):
    x, z = profile(liquid_bridge(gap, th, volume))
    ax.plot(x, z, color=color, label=f"θ = {th}°")
    ax.plot(-x, z, color=color)
ax.set_aspect("equal")
ax.set_xlim(-0.9, 0.9)
ax.set_ylim(-0.55, 0.55)
ax.set_xlabel("x")
ax.set_ylabel("z")
ax.legend(loc="center right", fontsize="small");

## Force against gap

Wetting liquids pull the beads together, the more strongly the smaller the
contact angle. Non-wetting liquids ($\theta \gtrsim 90°$) push them apart at small
gaps, but there the symmetric bridge is **unstable**: the Hessian has a pair of
negative eigenvalues, sideways shifts in $x$ and $y$, so the drop would rather
slide out of the gap. Such points are drawn hollow. (Each point is a fresh
bridge, about half a second each.)

In [ ]:
import warnings

gaps = np.linspace(0.0, 0.6, 7)
fig, ax = plt.subplots(figsize=(6, 3.5))
for th, color in zip((20, 60, 100), ("tab:blue", "tab:green", "tab:red")):
    forces, stable = [], []
    for g in gaps:
        with warnings.catch_warnings():   # "Hessian not positive definite": see above
            warnings.simplefilter("ignore", pyse.EvolverWarning)
            b = liquid_bridge(g, th, volume)
        # every Evolver is a handle to the one engine: use b before the next build
        forces.append(force(b))
        stable.append(b.eigen_counts().negative == 0)
    forces, stable = np.array(forces), np.array(stable)
    ax.plot(gaps, forces, "-", color=color, label=f"θ = {th}°")
    ax.plot(gaps[stable], forces[stable], "o", color=color)
    ax.plot(gaps[~stable], forces[~stable], "o", mfc="white", color=color)
ax.axhline(0, color="gray", lw=0.5)
ax.set_xlabel("gap")
ax.set_ylabel("force (attractive > 0)")
ax.set_title(f"liquid volume {volume}")
ax.legend();

## Exporting for finite elements

For a simulation of the liquid (flow inside the bridge, say), the liquid region
needs a **closed surface** and a **volume mesh**. Evolver's mesh is only the
liquid–gas interface: the wetted parts of the beads are integrals, not facets,
so the body's surface has two openings. `body_surfaces(cap=True)` closes each
opening with a cap on the constraint its rim lies on (Evolver projects the cap's
points, so this works for any constraint formula), and tells the caps apart.

Every `Evolver` object is a handle to the one engine, and the sweep above left
its last bridge there; build the one to export again:

In [ ]:
ev = liquid_bridge(gap, theta, volume)
surface = ev.body_surfaces(cap=True)[1]          # body 1, the liquid

a, b, cc = (surface.points[surface.cells[:, i]] for i in range(3))
area = np.linalg.norm(np.cross(b - a, cc - a), axis=1)/2
print("watertight:", surface.watertight, "| caps on constraints:", surface.cap_constraints)
print(f"enclosed volume {np.einsum('ij,ij->i', a, np.cross(b, cc)).sum()/6:.5f} (target {volume})")
print(f"liquid-gas area {area[surface.cap_ids == 0].sum():.5f} (Evolver {ev.total_area:.5f}), "
      f"wetted areas {area[surface.cap_ids == 1].sum():.4f}, {area[surface.cap_ids == 2].sum():.4f}")

`volume_mesh` fills the closed surface with tetrahedra using
[Gmsh](https://gmsh.info) (`pip install gmsh`), keeping the surface triangles as
they are. The physical groups: 1 for the liquid–gas facets, 2 and 3 for the caps
(cap 1 and 2), and the body id for the tetrahedra. A `.msh` path keeps the group
names; that is what `dolfinx.io.gmshio.read_from_msh` reads.

In [ ]:
import os, tempfile
import meshio

folder = tempfile.mkdtemp()
mesh = surface.volume_mesh(size=0.04, path=os.path.join(folder, "bridge.msh"))

tets, P = mesh.cells_dict["tetra"], mesh.points
v = np.einsum("ij,ij->i", P[tets[:, 1]] - P[tets[:, 0]],
              np.cross(P[tets[:, 2]] - P[tets[:, 0]], P[tets[:, 3]] - P[tets[:, 0]]))
print(f"{len(tets)} tetrahedra, volume {np.abs(v).sum()/6:.5f}")
groups = mesh.cell_data_dict["gmsh:physical"]
print("boundary triangles per group:", dict(zip(*np.unique(groups["triangle"], return_counts=True))))

For other FEM codes, meshio writes the tetrahedra and the tagged boundary
triangles separately, for example as XDMF:

In [ ]:
meshio.write(os.path.join(folder, "bridge.xdmf"),
             meshio.Mesh(P, [("tetra", tets)], cell_data={"region": [groups["tetra"]]}))
meshio.write(os.path.join(folder, "bridge_boundary.xdmf"),
             meshio.Mesh(P, [("triangle", mesh.cells_dict["triangle"])],
                         cell_data={"boundary": [groups["triangle"]]}))
print(sorted(f for f in os.listdir(folder)))

# a cut through the tetrahedra
grid = pv.from_meshio(meshio.Mesh(P, [("tetra", tets)]))
plotter = pv.Plotter()
plotter.add_mesh(grid.clip(normal="y", invert=False, crinkle=True), color="lightskyblue",
                 show_edges=True)
add_beads(plotter, gap, opacity=0.3)
plotter.camera_position = [(0, -2.6, 0.4), (0, 0, 0), (0, 0, 1)]
plotter.show()

## Notes and limits

* **Changing parameters on a relaxed bridge.** `ev.parameters["theta"] = ...` or
  `["gap"]` followed by `settle(ev)` works for small changes and for lowering the
  contact angle; large jumps, or raising the angle so that the contact line has to
  retreat, can leave folded facets at the contact line (the energy is still right,
  but the measured angle is not). `liquid_bridge(...)` builds afresh each time,
  which takes about half a second.
* **Stability.** A relaxed bridge is an equilibrium, not necessarily a stable
  one: check `ev.eigen_counts().negative`. Non-wetting bridges
  at small gaps are saddles (see above).
* **Rupture.** At large gaps and small volumes a bridge is unstable and would
  break; the mesh cannot follow a pinch-off, so the relaxation fails there (for
  example volume 0.05 at gap 0.5).
* **Unequal beads, gravity, …** follow the same pattern: another sphere radius or
  centre in the constraint formulas, `gravity` and body densities in the datafile
  header.